In [3]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from openai import OpenAI

# 1. Load environment variables from your .env file
# (Note: standard filename is ".env", not ".evn" — adjust the path if your file is elsewhere)
env_path = "/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env"
loaded = load_dotenv(dotenv_path=env_path, override=True)

# Optional fallback if running from a subdirectory and not using the hardcoded path above:
# if not loaded:
#     load_dotenv(dotenv_path=os.path.join("..", ".env"), override=True)

# 2. Verify OPENAI_API_KEY was loaded
assert os.environ.get("OPENAI_API_KEY"), (
    f"OPENAI_API_KEY not found. Check that it is set in {env_path}"
)

# The OpenAI client reads OPENAI_API_KEY from the environment automatically
client = OpenAI()

# Use a valid OpenAI model identifier (e.g., "gpt-4o-mini", "gpt-4.1-mini", or whichever model your account supports)
MODEL = "gpt-4o-mini"
print(f"Client ready. Using model: {MODEL}")

Client ready. Using model: gpt-4o-mini


In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def multiply_numbers(a: int, b: int) -> int:
    """Multiply two whole numbers together and return the product."""
    return a * b


@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city name."""
    return f"Sunny and 26°C in {city}"


@tool
def write_to_file(msg: str) -> str:
    """Perform a write operation to a file."""
    return f"Write operation to a file for '{msg}' completed successfully."


agent = create_agent(
    model=f"openai:{MODEL}",
    tools=[multiply_numbers, get_weather, write_to_file],
    system_prompt=(
        "You are a helpful assistant. "
        "Whenever you call a tool, first write a 1-sentence explanation of your "
        "tool selection logic (why this tool matches the request and why other tools do not), "
        "and then invoke the tool."
    ),
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I want to perform a write operation to a file with the text 'Hello, world!'.",
            }
        ]
    }
)

for i, msg in enumerate(result["messages"], start=1):
    if msg.type == "human":
        print(f"[{i}] [HUMAN]: {msg.content}")

    elif msg.type == "ai":
        if msg.content:
            print(f"[{i}] [AI]: {msg.content}")
        if getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                print(f"[{i}] [TOOL CALL]: {tc['name']} with args {tc['args']}")

    elif msg.type == "tool":
        # THIS is the actual return value from write_to_file
        print(f"[{i}] [TOOL RESULT <- {msg.name}]: {msg.content}")

[1] [HUMAN]: Can you tell me what is my score in my test?
[2] [AI]: I currently do not have access to information about your test scores or educational data. Please check with your educational institution or the relevant platform for your test results.
